# LeadFlowML DSL Demo

This notebook demonstrates an end-to-end **sales lead classification DSL** built with **textX** and exported into **CWL**. The workflow trains on historical CRM cases and classifies new sales cases.

Scenarios covered:
1. Parse the DSL into a Python object model
2. Generate CWL tools + workflow
3. Run the default logistic-regression pipeline
4. Compare an alternative random-forest scenario
5. Execute the generated CWL workflow


In [ ]:
from pathlib import Path
import json
import subprocess
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path('/mnt/data/LeadFlowML_Project')
import os, sys
os.chdir(ROOT)
sys.path.append(str(ROOT / 'src'))

from leadflow_common import parse_leadflow, validate_config, run_training, run_scoring
from dsl_processor import generate_all


## 1) Parse the default DSL scenario

In [ ]:
grammar = ROOT / 'dsl' / 'leadflow.tx'
default_dsl = ROOT / 'workflows' / 'lead_qualification.leadflow'
cfg = parse_leadflow(grammar, default_dsl)
pd.DataFrame([cfg.as_dict()]).T


## 2) Generate CWL files from the DSL

In [ ]:
generated_dir = ROOT / 'generated_cwl'
generate_all(grammar, default_dsl, generated_dir)
[p.name for p in sorted(generated_dir.iterdir())]


In [ ]:
print((generated_dir / 'workflow.cwl').read_text()[:1000])


## 3) Run the default local pipeline

In [ ]:
train_df = pd.read_csv(ROOT / cfg.train_path)
score_df = pd.read_csv(ROOT / cfg.score_path)
validation = validate_config(cfg, train_df, score_df)
validation


In [ ]:
report_default = run_training(cfg, ROOT)
preds_default = run_scoring(cfg, ROOT)
report_default


In [ ]:
preds_default.head(10)


## 4) Alternative scenario: random forest

In [ ]:
alt_dsl = ROOT / 'workflows' / 'lead_qualification_rf.leadflow'
cfg_rf = parse_leadflow(grammar, alt_dsl)
report_rf = run_training(cfg_rf, ROOT)
preds_rf = run_scoring(cfg_rf, ROOT)
report_rf


In [ ]:
comparison = pd.DataFrame({
    'metric': ['accuracy', 'precision', 'recall', 'f1', 'roc_auc', 'cv_f1_mean'],
    'logistic_regression': [
        report_default['metrics']['accuracy'],
        report_default['metrics']['precision'],
        report_default['metrics']['recall'],
        report_default['metrics']['f1'],
        report_default['metrics']['roc_auc'],
        report_default['metrics']['cv_f1_mean'],
    ],
    'random_forest': [
        report_rf['metrics']['accuracy'],
        report_rf['metrics']['precision'],
        report_rf['metrics']['recall'],
        report_rf['metrics']['f1'],
        report_rf['metrics']['roc_auc'],
        report_rf['metrics']['cv_f1_mean'],
    ]
})
comparison


In [ ]:
artifacts = ROOT / 'artifacts'
artifacts.mkdir(exist_ok=True)

ax = comparison.set_index('metric').plot(kind='bar', figsize=(10, 5), title='Model comparison on the sales lead scenario')
ax.set_ylabel('score')
ax.figure.tight_layout()
plt.savefig(artifacts / 'metrics_comparison.png', dpi=160)
plt.show()


In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(preds_default['qualified_probability'], bins=8)
plt.title('Probability distribution for new sales cases (default scenario)')
plt.xlabel('qualified_probability')
plt.ylabel('count')
plt.tight_layout()
plt.savefig(artifacts / 'prediction_distribution.png', dpi=160)
plt.show()


## 5) Execute the generated CWL workflow

In [ ]:
print('To execute the portable workflow, run:')
print('cd generated_cwl && cwltool workflow.cwl inputs.yml')
print('Current generated CWL files:')
print(sorted([p.name for p in generated_dir.iterdir()]))


In [ ]:
portable_outputs = ['validation_report.json', 'evaluation_report.json', 'new_case_predictions.csv']
{k: (generated_dir / k).exists() for k in portable_outputs}


## 6) Summary

- The DSL is concise enough for domain experts to read and modify.
- textX handles parsing + model creation.
- The processor turns the DSL into reusable CWL CommandLineTools and a Workflow.
- The same domain specification supports both **local execution** and **portable workflow execution** through CWL.
